# 01 · Exploración y preprocesamiento de datos
**Proyecto:** Predicción de deserción y éxito académico · **Asignatura:** Minería de Datos (UEA)

Contenido: carga → calidad de datos (nulos, duplicados, atípicos) → descriptivas → 6 visualizaciones → variables derivadas → preprocesamiento y partición 80/20.

> Si algún nombre de columna no coincide, la función `col()` te avisará: revisa `df.columns` y ajusta las palabras clave.

## 0. Configuración

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

ROOT = Path("..") if Path("../data").exists() else Path(".")
RAW, PROC = ROOT / "data/raw", ROOT / "data/processed"
FIG, TAB = ROOT / "reports/figures", ROOT / "reports/tables"
for p in (RAW, PROC, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

## 1. Carga de datos

In [ ]:
csv_path = RAW / "dropout.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path, sep=None, engine="python")   # detecta ; o ,
else:
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=697)                               # UCI: Predict Students' Dropout and Academic Success
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    df.to_csv(csv_path, index=False)

df.columns = [c.strip() for c in df.columns]
print("Registros y columnas:", df.shape)
df.head()

In [ ]:
def col(*keys):
    """Busca una columna cuyo nombre contenga todas las palabras clave (sin distinguir mayúsculas)."""
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keys):
            return c
    raise KeyError(f"No se encontró columna con {keys}. Revisa df.columns")

TARGET = col("target")

# Variables que son CÓDIGOS (categóricas) aunque vengan como enteros
CAT_KEYS = ["marital", "application mode", "application order", "course", "attendance",
            "previous qualification", "nacionality", "nationality", "mother's", "father's",
            "displaced", "special needs", "debtor", "tuition", "gender", "scholarship", "international"]
cat_cols = [c for c in df.columns if c != TARGET and any(k in c.lower() for k in CAT_KEYS)
            and "grade" not in c.lower()]
num_cols = [c for c in df.columns if c != TARGET and c not in cat_cols]
print(f"{len(cat_cols)} categóricas · {len(num_cols)} numéricas")

## 2. Calidad de datos

In [ ]:
# 2.1 Porcentaje de nulos por variable
nulos = (df.isna().mean() * 100).round(2).rename("% nulos").to_frame()
nulos.to_csv(TAB / "nulos_por_variable.csv")
print("Variables con nulos:", int((nulos["% nulos"] > 0).sum()), "de", len(nulos))
nulos.sort_values("% nulos", ascending=False).head(10)

In [ ]:
# 2.2 Duplicados: cantidad y porcentaje
dup_n = int(df.duplicated().sum())
dup_p = round(dup_n / len(df) * 100, 2)
print(f"Registros duplicados: {dup_n} ({dup_p} %)")

In [ ]:
# 2.3 Valores atípicos (regla IQR) en variables numéricas
filas = []
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    filas.append({"variable": c, "atipicos": n_out, "% atipicos": round(n_out / len(df) * 100, 2)})
atipicos = pd.DataFrame(filas).sort_values("atipicos", ascending=False)
atipicos.to_csv(TAB / "atipicos_iqr.csv", index=False)
atipicos.head(10)

**Decisión sobre atípicos (completar):** explica aquí si los conservas o los tratas, y por qué.
Ejemplo: una edad alta al ingreso es un caso legítimo (estudiante adulto), no un error de captura.

## 3. Estadísticas descriptivas

In [ ]:
desc = df[num_cols].describe().T.round(2)
desc.to_csv(TAB / "estadisticas_descriptivas.csv")
display(desc.head(15))

dist = df[TARGET].value_counts().to_frame("n")
dist["%"] = (dist["n"] / len(df) * 100).round(2)
dist

## 4. Visualizaciones

In [ ]:
age, adm = col("age"), col("admission", "grade")
debtor, tuition, schol = col("debtor"), col("tuition"), col("scholarship")
appr1 = col("1st sem", "approved")
orden = sorted(df[TARGET].unique())

# 1. Distribución de la clase objetivo
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=TARGET, order=orden, ax=ax)
ax.set_title("Distribución de la variable objetivo"); ax.set_xlabel(""); ax.set_ylabel("Estudiantes")
fig.savefig(FIG / "01_distribucion_objetivo.png", dpi=150, bbox_inches="tight"); plt.show()

# 2. Edad al ingreso por clase
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x=age, hue=TARGET, hue_order=orden, bins=30, multiple="stack", ax=ax)
ax.set_title("Edad al ingreso por clase")
fig.savefig(FIG / "02_edad_por_clase.png", dpi=150, bbox_inches="tight"); plt.show()

# 3. Nota de admisión por clase
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x=TARGET, y=adm, order=orden, ax=ax)
ax.set_title("Nota de admisión por clase"); ax.set_xlabel("")
fig.savefig(FIG / "03_nota_admision.png", dpi=150, bbox_inches="tight"); plt.show()

# 4. Tasa de resultado según factores financieros
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, c in zip(axes, [debtor, tuition, schol]):
    pd.crosstab(df[c], df[TARGET], normalize="index").mul(100).plot(kind="bar", stacked=True, ax=ax, legend=False)
    ax.set_title(c); ax.set_xlabel(""); ax.set_ylabel("%")
axes[-1].legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(FIG / "04_factores_financieros.png", dpi=150, bbox_inches="tight"); plt.show()

# 5. Unidades aprobadas en el 1.er semestre por clase
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x=TARGET, y=appr1, order=orden, ax=ax)
ax.set_title("Unidades aprobadas, 1.er semestre"); ax.set_xlabel("")
fig.savefig(FIG / "05_aprobadas_1s.png", dpi=150, bbox_inches="tight"); plt.show()

# 6. Mapa de calor de correlación (numéricas)
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre variables numéricas")
fig.savefig(FIG / "06_correlacion.png", dpi=150, bbox_inches="tight"); plt.show()

**Interpretación (completar):** escribe 1–2 frases por gráfico con lo que observas; irán al documento y al artículo.

## 5. Variables derivadas (feature engineering)

In [ ]:
e1, a1, g1 = col("1st sem", "enrolled"), col("1st sem", "approved"), col("1st sem", "grade")
e2, a2, g2 = col("2nd sem", "enrolled"), col("2nd sem", "approved"), col("2nd sem", "grade")

df["tasa_aprob_1s"] = (df[a1] / df[e1].replace(0, np.nan)).fillna(0)      # aprobadas / inscritas
df["tasa_aprob_2s"] = (df[a2] / df[e2].replace(0, np.nan)).fillna(0)
df["variacion_nota"] = df[g2] - df[g1]                                    # mejora o caída entre semestres
df["total_aprobadas"] = df[a1] + df[a2]
df["riesgo_financiero"] = ((df[debtor] == 1) | (df[tuition] == 0)).astype(int)  # deudor o matrícula no al día

num_cols += ["tasa_aprob_1s", "tasa_aprob_2s", "variacion_nota", "total_aprobadas"]
cat_cols += ["riesgo_financiero"]
df[["tasa_aprob_1s", "tasa_aprob_2s", "variacion_nota", "total_aprobadas", "riesgo_financiero"]].describe().round(2)

## 6. Preprocesamiento y partición
Se **codifica** (one-hot) y se **normaliza** (StandardScaler) con un `ColumnTransformer` que se ajusta **solo con el entrenamiento** (evita fuga de datos).

In [ ]:
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
Xtr_t = prep.fit_transform(X_tr)
Xte_t = prep.transform(X_te)
print(f"Entrenamiento: {X_tr.shape[0]} (80 %) · Prueba: {X_te.shape[0]} (20 %)")
print(f"Dimensión tras codificación: {Xtr_t.shape[1]} columnas")
print(y_tr.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Escenario "solo datos de ingreso" (detección temprana) para comparar luego en el modelado
cols_ingreso = [c for c in X.columns if "curricular" not in c.lower()
                and c not in ["tasa_aprob_1s", "tasa_aprob_2s", "variacion_nota", "total_aprobadas"]]
print(len(cols_ingreso), "columnas disponibles al momento del ingreso")

# Guardar datos con variables derivadas para los siguientes notebooks
df.to_csv(PROC / "dropout_procesado.csv", index=False)

**Siguiente paso:** `02_modelado.ipynb` (árbol de decisión, Random Forest, regresión logística dentro de un `Pipeline` con `prep`, CV de 5 particiones) y `03_evaluacion.ipynb`.